# 面试问题：Medusa 多解码头、候选树、Tree Attention、Target Verification 与 KV 精确提交怎样实现？

**一句话回答。** Medusa 在目标大模型最后一个隐藏状态上增加若干轻量解码头，让第 \(k\) 个头并行猜测更远的第 \(k+2\) 个 token；随后把各头的 top-k 结果组合成共享前缀的候选树，用只允许节点看见其祖先的 tree attention 在一次目标模型前向中验证所有分支，选择被目标模型接受的最长前缀，最后只把该前缀对应的 KV 按连续位置提交，拒绝分支的临时 KV 必须丢弃。

本 Notebook 使用 PyTorch 基础张量运算从零搭建一个可执行的小模型，重点证明形状、可见性、验收和缓存状态机。它不是生产推理内核，也不声称这里的玩具转移矩阵具备语言能力。

**论文入口：** [Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads](https://arxiv.org/abs/2401.10774)。论文将一次解码拆成候选生成、tree attention 并行处理和候选验收三个阶段，并强调多头共享目标模型隐藏状态，不需要维护独立 draft model。

In [ ]:
import itertools  # 导入笛卡尔积工具，用于显式构造多头候选组合。
import math  # 导入数学工具，用于计算注意力缩放系数。
import torch  # 导入 PyTorch 基础张量与自动求导能力。
torch.manual_seed(235)  # 固定随机种子，使教学断言可以稳定复现。
hidden_size = 8  # 设置玩具目标模型最后隐藏层的维度。
vocab_size = 10  # 设置玩具词表大小，便于完整检查每个候选 token。
medusa_head_count = 3  # 设置三个额外解码头，分别预测更远的三个位置。
assert hidden_size > 0  # 验证隐藏维度是有效正整数。
assert vocab_size == 10  # 验证后续人工转移矩阵与词表配置一致。
assert medusa_head_count == 3  # 验证候选树深度与额外解码头数量一致。
assert torch.initial_seed() == 235  # 验证本次实验确实使用了固定随机种子。

## 1. 多解码头究竟预测什么？

普通 LM head 根据位置 \(t\) 的隐藏状态预测 \(t+1\)；第 0 个 Medusa 额外头预测 \(t+2\)，第 1 个预测 \(t+3\)，以此类推。它们不是依次自回归运行，而是读取同一个 \(h_t\) 并行给出边缘分布，因此很快，但越远的头通常越不准，而且不同位置的预测并非独立。

论文中的头是带残差的一层前馈结构：先做线性映射和 SiLU，再与输入隐藏状态相加，最后映射到词表。下面不用 nn.Linear，直接声明参数和矩阵乘法；把第一层初始化为零、输出矩阵复制目标 LM head，可让新增头初始时与原始分布对齐。

In [ ]:
class ManualMedusaHead(torch.nn.Module):  # 定义只依赖基础参数和矩阵乘法的 Medusa 解码头。
    def __init__(self, width, vocabulary, target_weight):  # 接收隐藏维度、词表大小和目标头权重。
        super().__init__()  # 初始化 PyTorch 模块基类以注册可训练参数。
        self.w1 = torch.nn.Parameter(torch.zeros(width, width))  # 将残差前馈层初始化为零以对齐初始分布。
        self.b1 = torch.nn.Parameter(torch.zeros(width))  # 将前馈偏置初始化为零避免引入初始漂移。
        self.w2 = torch.nn.Parameter(target_weight.detach().clone())  # 复制目标 LM head 权重作为词表投影初值。
        self.vocabulary = vocabulary  # 保存词表大小以便检查输出数据合同。
    def forward(self, hidden):  # 定义隐藏状态到候选 token logits 的前向过程。
        projected = hidden @ self.w1 + self.b1  # 手工执行第一层仿射映射。
        activated = projected * torch.sigmoid(projected)  # 用基础算子实现 SiLU 激活函数。
        residual = hidden + activated  # 将激活结果与原隐藏状态相加形成残差连接。
        return residual @ self.w2  # 手工投影到完整词表并返回未归一化 logits。
target_lm_weight = torch.randn(hidden_size, vocab_size) / math.sqrt(hidden_size)  # 构造玩具目标 LM head 权重并控制方差。
hidden_states = torch.randn(2, 4, hidden_size)  # 构造两个样本、四个时间步的目标模型隐藏状态。
medusa_heads = torch.nn.ModuleList([ManualMedusaHead(hidden_size, vocab_size, target_lm_weight) for _ in range(medusa_head_count)])  # 注册三个可独立训练的额外解码头。
base_logits = hidden_states @ target_lm_weight  # 用同一隐藏状态计算原始目标 LM head 的 logits。
initial_head_logits = [head(hidden_states) for head in medusa_heads]  # 并行概念下逐头计算三个未来位置分布。
assert base_logits.shape == (2, 4, vocab_size)  # 验证目标头输出满足批次、时间和词表三维合同。
assert len(initial_head_logits) == medusa_head_count  # 验证每个预测偏移都有一个对应输出。
assert all(logits.shape == base_logits.shape for logits in initial_head_logits)  # 验证所有额外头的输出形状完全一致。
assert all(torch.allclose(logits, base_logits) for logits in initial_head_logits)  # 验证零初始化残差使所有新增头初始对齐目标头。

## 2. 训练标签必须按预测距离正确移位

最常见实现错误是让所有头都学习 next token。若当前隐藏状态覆盖位置 \(t\)，第一个额外头的标签应从 \(t+2\) 开始，第三个额外头则从 \(t+4\) 开始。Medusa-1 冻结 backbone，只训练这些头；越远的标签更不确定，通常使用随距离衰减的权重。Medusa-2 会联合训练 backbone，但还要保留原 LM loss、采用差分学习率和 head warmup，避免破坏原模型能力。

下面手写稳定版 log-softmax 与交叉熵，不调用高层 loss。这个训练片段只展示标签和梯度路径；真正训练必须处理 padding mask、序列边界、混合精度以及分布式归约。

In [ ]:
token_batch = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8], [2, 3, 4, 5, 6, 7, 8, 9]])  # 构造两条足够长的监督 token 序列。
training_steps = hidden_states.shape[1]  # 读取每条隐藏状态序列可参与训练的时间步数。
def shifted_future_labels(tokens, steps, ahead):  # 定义按未来距离切取监督标签的函数。
    return tokens[:, ahead:ahead + steps]  # 返回与隐藏时间维对齐但向未来移动的标签窗口。
def manual_cross_entropy(logits, labels):  # 定义不依赖高层损失函数的交叉熵。
    stabilized = logits - logits.max(dim=-1, keepdim=True).values  # 减去最大值以避免指数上溢。
    log_normalizer = torch.log(torch.exp(stabilized).sum(dim=-1, keepdim=True))  # 手工计算稳定的对数配分函数。
    log_probabilities = stabilized - log_normalizer  # 得到每个词表项的对数概率。
    selected = torch.gather(log_probabilities, -1, labels.unsqueeze(-1)).squeeze(-1)  # 取出真实未来 token 的对数概率。
    return -selected.mean()  # 对批次和时间维求平均得到标量负对数似然。
head_losses = []  # 创建列表收集不同预测距离的独立损失。
for head_index, head in enumerate(medusa_heads):  # 逐个额外头建立正确的移位监督信号。
    labels = shifted_future_labels(token_batch, training_steps, head_index + 2)  # 让第零个额外头从未来第二个 token 开始学习。
    logits = head(hidden_states.detach())  # 冻结教学 backbone 隐藏状态并只训练额外头。
    head_losses.append(manual_cross_entropy(logits, labels))  # 计算当前预测距离的手写交叉熵。
loss_weights = [0.8 ** (index + 1) for index in range(medusa_head_count)]  # 给更远且更难的预测施加递减权重。
weighted_medusa_loss = sum(weight * loss for weight, loss in zip(loss_weights, head_losses))  # 汇总 Medusa-1 的多头训练目标。
weighted_medusa_loss.backward()  # 反向传播以验证额外头参数拥有真实梯度路径。
assert shifted_future_labels(token_batch, training_steps, 2).tolist() == [[3, 4, 5, 6], [4, 5, 6, 7]]  # 验证第一个额外头学习未来第二个 token。
assert shifted_future_labels(token_batch, training_steps, 4).tolist() == [[5, 6, 7, 8], [6, 7, 8, 9]]  # 验证第三个额外头学习未来第四个 token。
assert len(head_losses) == medusa_head_count  # 验证每个额外头都贡献了独立损失。
assert torch.isfinite(weighted_medusa_loss)  # 验证手写数值稳定交叉熵没有产生无穷或非数。
assert medusa_heads[0].w1.grad is not None  # 验证冻结隐藏状态后梯度仍能到达新增解码头。

## 3. 从各头 top-k 构造候选树

一次 Medusa step 通常先由原始 LM head 产生下一个 token，再从各额外头保留若干高分 token。最直观的候选集合是各层选择的笛卡尔积。若三个头各保留 2 个 token，就得到 8 条长度为 4 的候选，但这些候选拥有大量相同前缀。

工程上不能无脑增大分支数：完整树节点量是各层前缀乘积之和，更多节点提高命中机会，也增加目标模型验证算力。生产系统可根据校准集估计不同“头序号 × rank”的准确率，在固定节点预算下选不规则树。下面用受控 logits 保证既有正确路径，也有会被拒绝的兄弟路径。

In [ ]:
root_logits = torch.tensor([0.1, 0.2, 3.4, 0.0, 0.1, 0.2, 0.0, 0.1, 0.0, 0.0])  # 构造原始 LM head 对第一个 token 的受控预测。
proposal_logits = [torch.tensor([0.0, 0.2, 0.4, 3.2, 0.1, 0.0, 0.0, 2.8, 0.1, 0.0]), torch.tensor([0.0, 0.1, 0.0, 0.2, 3.1, 0.0, 0.0, 0.1, 2.7, 0.0]), torch.tensor([0.0, 2.6, 0.1, 0.0, 0.2, 3.0, 0.0, 0.0, 0.1, 0.0])]  # 构造三个额外头的受控未来位置 logits。
branch_widths = [2, 2, 2]  # 规定每个额外头只保留概率最高的两个 token。
root_token = int(torch.argmax(root_logits).item())  # 选择目标 LM head 的贪心根 token。
head_choices = [torch.topk(logits, width).indices.tolist() for logits, width in zip(proposal_logits, branch_widths)]  # 提取每个未来位置的 top-k 候选。
candidate_tails = list(itertools.product(*head_choices))  # 计算各未来位置选择的笛卡尔积。
candidates = [(root_token,) + tail for tail in candidate_tails]  # 给每条候选尾部补上原始 LM head 的根 token。
def candidate_score(sequence):  # 定义只用于同验收长度下排序的提议分数。
    return float(root_logits[sequence[0]] + sum(proposal_logits[index][sequence[index + 1]] for index in range(medusa_head_count)))  # 累加各头对应 token 的原始 logits。
ordered_candidates = sorted(candidates, key=candidate_score, reverse=True)  # 按提议分数从高到低排列完整候选。
assert root_token == 2  # 验证原始 LM head 选择了预设根 token。
assert head_choices == [[3, 7], [4, 8], [5, 1]]  # 验证每层 top-k 分支与设计一致。
assert len(candidates) == 8  # 验证三个二分层生成二的三次方条候选。
assert all(len(candidate) == 4 for candidate in candidates)  # 验证每条候选含一个根 token 和三个未来 token。
assert ordered_candidates[0] == (2, 3, 4, 5)  # 验证全取 rank-1 时提议分数最高。

## 4. 前缀去重与 Tree Attention Mask

若把 8 条候选当成 batch，会重复计算共同的 token 2、共同的 token 3 或 7。候选树用“父节点与 token”作为唯一边键合并公共前缀：本例原本有 \(8×4=32\) 个候选位置，去重后只剩 \(1+2+4+8=15\) 个节点。

Tree attention 的安全条件是：每个节点可以看见 prompt 和自己路径上的祖先，但绝不能看到兄弟、堂兄弟或未来节点。位置编码也不能使用扁平数组下标，而要使用 prompt 长度加 depth；同一深度的不同分支共享逻辑位置。mask 写反会导致候选之间互相泄露，让 target verification 得到虚假的高接受率。

In [ ]:
def build_prefix_tree(candidate_sequences):  # 定义按共享前缀压缩候选集合的建树函数。
    nodes = []  # 保存每个唯一候选节点的 token、父节点与深度。
    edge_to_node = {}  # 使用父节点和 token 组成的边键完成前缀去重。
    path_by_sequence = {}  # 保存每条完整候选对应的树节点索引路径。
    for sequence in candidate_sequences:  # 逐条插入候选并复用已经存在的公共前缀。
        parent = -1  # 使用负一表示直接连接到 prompt 的虚拟根。
        node_path = []  # 创建当前候选的实际节点索引路径。
        for depth, token in enumerate(sequence):  # 从根 token 到最远 token 逐层处理。
            edge = (parent, token)  # 用父节点身份区分不同上下文中的相同 token。
            if edge not in edge_to_node:  # 只为尚未出现的前缀边创建新节点。
                node_id = len(nodes)  # 使用当前节点数量分配连续节点索引。
                edge_to_node[edge] = node_id  # 记录该前缀边对应的唯一节点。
                nodes.append({"id": node_id, "token": token, "parent": parent, "depth": depth})  # 保存树注意力和验证所需的节点元数据。
            parent = edge_to_node[edge]  # 将当前节点设为下一层 token 的父节点。
            node_path.append(parent)  # 把当前唯一节点加入该候选路径。
        path_by_sequence[sequence] = node_path  # 完整记录候选到树节点序列的映射。
    return nodes, path_by_sequence  # 返回压缩后的树和候选路径索引。
tree_nodes, candidate_node_paths = build_prefix_tree(candidates)  # 将八条候选压缩为共享前缀树。
tree_mask = torch.zeros(len(tree_nodes), len(tree_nodes), dtype=torch.bool)  # 创建默认全部不可见的树内注意力矩阵。
for node in tree_nodes:  # 为每个查询节点沿父指针标记自身和全部祖先。
    cursor = node["id"]  # 从当前节点开始回溯祖先链。
    while cursor >= 0:  # 在到达连接 prompt 的虚拟根前持续回溯。
        tree_mask[node["id"], cursor] = True  # 允许当前节点关注路径上的该祖先节点。
        cursor = tree_nodes[cursor]["parent"]  # 移动到更上一层父节点继续标记。
prompt_length = 3  # 假设目标模型已经缓存三个 prompt token。
tree_position_ids = torch.tensor([prompt_length + node["depth"] for node in tree_nodes])  # 按树深度而非扁平节点编号分配位置。
correct_path = candidate_node_paths[(2, 3, 4, 5)]  # 取出预设正确候选的树节点路径。
sibling_path = candidate_node_paths[(2, 7, 4, 5)]  # 取出在第二层分叉的兄弟候选路径。
assert len(tree_nodes) == 15  # 验证前缀共享把三十二个位置压缩成十五个节点。
assert bool(torch.all(torch.diag(tree_mask)))  # 验证每个节点都能够关注自身。
assert bool(tree_mask[correct_path[-1], correct_path[0]])  # 验证叶子节点能够关注同路径根节点。
assert not bool(tree_mask[correct_path[1], sibling_path[1]])  # 验证不同第二层分支之间完全不可见。
assert tree_position_ids[correct_path].tolist() == [3, 4, 5, 6]  # 验证正确路径的位置编号按深度连续递增。
assert tree_position_ids[correct_path[1]] == tree_position_ids[sibling_path[1]]  # 验证同深度兄弟节点共享同一逻辑位置。

## 5. 用基础算子执行一次 Tree Attention

目标模型对所有树节点做一次前向时，prompt KV 对每个节点都可见；树内 KV 则由上面的祖先 mask 控制。下面手写缩放点积、mask、softmax 和加权求和。真实模型还会执行多头拆分、RoPE、层归一化、残差、张量并行和 fused kernel，但可见性不变量相同。

注意 mask 的行表示 query 节点、列表示 key 节点。屏蔽值要在 softmax 前写入极小数；若在 softmax 后简单乘零却不重新归一化，权重和不再等于 1。所有 prompt 列都保留，而被屏蔽的兄弟树节点权重必须严格为零。

In [ ]:
attention_width = 6  # 设置单个教学注意力头的向量维度。
prompt_keys = torch.randn(prompt_length, attention_width)  # 构造已经提交的 prompt key 缓存。
prompt_values = torch.randn(prompt_length, attention_width)  # 构造已经提交的 prompt value 缓存。
tree_queries = torch.randn(len(tree_nodes), attention_width)  # 构造所有候选树节点的 query。
tree_keys = torch.randn(len(tree_nodes), attention_width)  # 构造所有候选树节点的临时 key。
tree_values = torch.randn(len(tree_nodes), attention_width)  # 构造所有候选树节点的临时 value。
def manual_tree_attention(queries, prefix_keys, prefix_values, candidate_keys, candidate_values, allowed_tree):  # 定义带 prompt 与祖先可见性的手写树注意力。
    scale = math.sqrt(queries.shape[-1])  # 计算缩放点积注意力的维度归一化因子。
    prefix_scores = queries @ prefix_keys.transpose(0, 1) / scale  # 计算每个树节点对全部 prompt key 的分数。
    tree_scores = queries @ candidate_keys.transpose(0, 1) / scale  # 计算树节点之间尚未屏蔽的两两分数。
    blocked_value = torch.finfo(tree_scores.dtype).min  # 取得当前浮点类型可表示的极小屏蔽值。
    masked_tree_scores = tree_scores.masked_fill(~allowed_tree, blocked_value)  # 屏蔽兄弟、未来和其他分支节点。
    combined_scores = torch.cat([prefix_scores, masked_tree_scores], dim=-1)  # 拼接始终可见的 prompt 列与受控树节点列。
    stable_scores = combined_scores - combined_scores.max(dim=-1, keepdim=True).values  # 在指数运算前减去每行最大值保证稳定。
    unnormalized = torch.exp(stable_scores)  # 将稳定 logits 转换为非归一化注意力权重。
    weights = unnormalized / unnormalized.sum(dim=-1, keepdim=True)  # 手工归一化每个查询节点的全部可见权重。
    combined_values = torch.cat([prefix_values, candidate_values], dim=0)  # 按分数列顺序拼接 prompt 与候选 value。
    return weights @ combined_values, weights  # 返回树节点上下文表示和可审计注意力矩阵。
tree_context, tree_weights = manual_tree_attention(tree_queries, prompt_keys, prompt_values, tree_keys, tree_values, tree_mask)  # 在一次批量张量计算中处理全部候选节点。
assert tree_context.shape == (15, attention_width)  # 验证每个压缩树节点都得到一个上下文向量。
assert tree_weights.shape == (15, prompt_length + 15)  # 验证注意力列同时覆盖 prompt 和完整候选树。
assert torch.allclose(tree_weights.sum(dim=-1), torch.ones(15))  # 验证手写 softmax 后每行权重严格归一化。
assert tree_weights[correct_path[1], prompt_length + sibling_path[1]].item() == 0.0  # 验证正确分支不会读取兄弟分支的 value。
assert torch.isfinite(tree_context).all()  # 验证屏蔽与归一化没有产生非数值结果。

## 6. Target Verification：提议头没有最终决定权

Medusa head 只负责 proposal，真正验收必须读取原始目标模型在对应父前缀下产生的 logits。Tree attention 保证每个节点的隐藏状态只依赖自己的祖先，因此一次目标前向即可为 15 个节点同时给出“该位置应当是什么”的判断。随后沿每条候选从根开始比较；一旦某个 token 不被接受，其后缀全部无效。

为清楚证明状态机，下面把目标模型简化为确定性 token 转移矩阵，并采用贪心验收。这样候选 2、3、4、5 与逐 token 目标模型完全一致。采样场景若要求输出分布严格等价，需要规范的 rejection sampling；论文的 typical acceptance 可以增加接受长度，但它只追求相近生成质量，不能宣称与目标分布完全相同。

In [ ]:
transition_logits = torch.full((vocab_size, vocab_size), -9.0)  # 创建玩具目标模型的确定性转移 logits 矩阵。
target_transitions = {0: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 6, 8: 9, 1: 1, 9: 0}  # 定义不同前缀末 token 下目标模型的贪心下一个 token。
for previous_token, next_token in target_transitions.items():  # 逐行写入每个上下文唯一的高分目标 token。
    transition_logits[previous_token, next_token] = 9.0  # 将预设目标 token 的 logit 提升为该行最大值。
prompt_last_token = 0  # 假设 prompt 最后一个 token 是零并作为候选根的验证上下文。
verified_target_by_node = {}  # 创建映射保存目标模型对每个树节点位置的贪心判断。
for node in tree_nodes:  # 利用父节点路径为全部候选节点批量建立验证标签。
    parent_id = node["parent"]  # 读取当前候选节点在前缀树中的父节点。
    previous_token = prompt_last_token if parent_id < 0 else tree_nodes[parent_id]["token"]  # 根节点读取 prompt，其他节点读取各自父前缀末 token。
    verified_target_by_node[node["id"]] = int(torch.argmax(transition_logits[previous_token]).item())  # 保存目标模型在该前缀下认可的 token。
def accepted_prefix_length(sequence, node_path):  # 定义从根开始计算连续通过验证的前缀长度。
    accepted = 0  # 将当前候选的已接受 token 数初始化为零。
    for token, node_id in zip(sequence, node_path):  # 按生成顺序比较提议 token 与目标模型判断。
        if token != verified_target_by_node[node_id]:  # 检查当前 token 是否偏离目标模型在同一父前缀下的选择。
            break  # 首次拒绝后立即停止，禁止跳过错误 token 验收后缀。
        accepted += 1  # 只有当前 token 匹配时才扩展连续接受前缀。
    return accepted  # 返回可以安全提交的连续 token 数量。
acceptance_lengths = {sequence: accepted_prefix_length(sequence, candidate_node_paths[sequence]) for sequence in candidates}  # 同时计算所有候选的连续验收长度。
best_candidate = max(candidates, key=lambda sequence: (acceptance_lengths[sequence], candidate_score(sequence)))  # 优先选择接受最长且提议分数更高的候选。
assert len(verified_target_by_node) == len(tree_nodes)  # 验证每个压缩树节点都获得了目标模型判断。
assert acceptance_lengths[(2, 3, 4, 5)] == 4  # 验证正确路径的四个 token 全部被目标模型接受。
assert acceptance_lengths[(2, 7, 4, 5)] == 1  # 验证第二个 token 错误时只有根 token 能被接受。
assert max(acceptance_lengths.values()) == 4  # 验证本轮能够一次安全推进四个自回归位置。
assert best_candidate == (2, 3, 4, 5)  # 验证最长连续前缀规则选中了目标模型一致路径。

## 7. KV 精确提交：树缓存和序列缓存不是同一布局

Tree attention 前向产生的 15 组 KV 是**临时候选缓存**，其扁平顺序是树节点编号，并不等于最终 token 序列位置。验收后要按胜出路径的节点索引 gather，只追加前 accepted length 个节点，并把它们映射到连续的 prompt 后续位置。所有未被选中的兄弟 KV 以及胜出候选中首个拒绝 token 之后的 KV 都必须丢弃。

若错误地把整棵树追加到普通 KV cache，下一轮会把互斥分支当成真实历史；若只修改逻辑长度但复用错误物理槽位，也可能造成位置编码、连续批处理 page table 或 prefix cache 污染。生产实现应把“暂存 arena、提交索引、逻辑长度和 page ownership”放在同一个原子事务里。

In [ ]:
accepted_count = acceptance_lengths[best_candidate]  # 读取胜出候选可连续提交的 token 数量。
accepted_node_ids = candidate_node_paths[best_candidate][:accepted_count]  # 只截取胜出路径中真正通过验证的树节点。
prompt_keys_snapshot = prompt_keys.clone()  # 保存提交前的 prompt key 以检查只追加不篡改。
prompt_values_snapshot = prompt_values.clone()  # 保存提交前的 prompt value 以检查历史缓存不变。
committed_keys = torch.cat([prompt_keys, tree_keys[accepted_node_ids]], dim=0)  # 按胜出路径顺序收集并追加候选 key。
committed_values = torch.cat([prompt_values, tree_values[accepted_node_ids]], dim=0)  # 按同一索引顺序收集并追加候选 value。
committed_position_ids = torch.arange(prompt_length, prompt_length + accepted_count)  # 为提交 token 分配连续的真实序列位置。
rejected_node_ids = set(range(len(tree_nodes))) - set(accepted_node_ids)  # 明确记录本轮不得进入持久缓存的其他树节点。
assert committed_keys.shape[0] == prompt_length + accepted_count  # 验证持久 key 长度只增长实际接受的 token 数。
assert committed_values.shape == committed_keys.shape  # 验证 key 与 value 的缓存长度和向量维完全一致。
assert torch.equal(committed_keys[prompt_length:], tree_keys[accepted_node_ids])  # 验证追加区严格来自胜出路径而非树的扁平前缀。
assert sibling_path[1] in rejected_node_ids and sibling_path[1] not in accepted_node_ids  # 验证被拒绝兄弟分支没有污染提交索引。
assert torch.equal(prompt_keys, prompt_keys_snapshot) and torch.equal(prompt_values, prompt_values_snapshot)  # 验证提交操作没有原地修改既有 prompt 历史。
assert committed_position_ids.tolist() == tree_position_ids[accepted_node_ids].tolist()  # 验证胜出路径的树深度位置可无歧义压紧为连续位置。

## 8. 端到端正确性、收益指标与上线验收

Medusa 的收益不是“多头一次输出了几个 token”，而是“每次昂贵目标模型前向最终提交了几个 token”。应统计平均接受长度、各深度接受率、tree 节点数、验证耗时、KV gather 耗时、吞吐和首 token 延迟；还要按 batch size、prompt/decode 比例、模型并行方式和采样温度分桶。树越大未必越快，因为验证计算会从内存带宽受限逐渐转向算力受限。

正确性回归必须把 Medusa 结果与关闭 Medusa 的目标模型逐 token 基线比较。贪心模式应逐 token 完全一致；分布保持采样需要统计检验与固定 RNG 语义；还要覆盖 EOS 位于候选中间、所有额外头首层即失败、动态 batch 插入或退出、paged KV 跨页、量化头、张量并行和异常回滚。

In [ ]:
def target_greedy_rollout(last_token, steps):  # 定义关闭 Medusa 时目标模型逐 token 运行的基线。
    generated = []  # 创建列表保存串行目标模型生成结果。
    current = last_token  # 从 prompt 最后一个真实 token 开始自回归。
    for _ in range(steps):  # 严格执行与接受长度相同次数的串行解码。
        current = int(torch.argmax(transition_logits[current]).item())  # 根据当前真实前缀选择目标模型贪心 token。
        generated.append(current)  # 将新 token 追加到基线真实历史。
    return generated  # 返回可与 Medusa 提交序列逐项比较的结果。
serial_baseline = target_greedy_rollout(prompt_last_token, accepted_count)  # 生成关闭 Medusa 时的四步串行基线。
medusa_committed_tokens = list(best_candidate[:accepted_count])  # 提取本轮经过 target verification 的实际提交 token。
metrics = {"candidate_count": len(candidates), "tree_node_count": len(tree_nodes), "target_passes": 1, "accepted_tokens": accepted_count, "tokens_per_target_pass": float(accepted_count)}  # 汇总本次受控实验的关键吞吐代理指标。
assert medusa_committed_tokens == serial_baseline  # 验证贪心 Medusa 结果与目标模型串行基线逐 token 完全相同。
assert metrics["candidate_count"] == 8  # 验证指标正确记录并行评估的完整候选数量。
assert metrics["tree_node_count"] < metrics["candidate_count"] * len(best_candidate)  # 验证共享前缀确实减少了重复验证节点。
assert metrics["target_passes"] == 1  # 验证教学流程用一次树形目标前向表达一轮并行验证。
assert metrics["tokens_per_target_pass"] == 4.0  # 验证本例一次昂贵目标前向安全推进了四个 token。
assert all(node_id not in accepted_node_ids for node_id in rejected_node_ids)  # 验证提交集合与拒绝集合保持严格互斥。

## 面试总结

一个完整回答应明确五个边界：第一，多头只是从同一隐藏状态并行预测不同未来偏移，不能当作独立目标模型；第二，top-k 笛卡尔积要压成共享前缀树，并在节点预算下权衡命中率和验证成本；第三，tree attention 只能看 prompt、自己和祖先，位置编号按深度设置；第四，目标模型在正确父前缀下逐节点验收，首个失败后的整段后缀都不能提交；第五，树形 KV 是暂存布局，最终缓存必须 gather 胜出前缀并压紧为连续序列。

还应主动说明质量口径：贪心精确验证可以和基线逐 token 对齐；严格分布等价采样需要 rejection sampling；typical acceptance 是质量—速度折中。最后用平均接受长度、目标前向次数、树节点预算、KV gather 成本和端到端延迟做真实硬件评测，而不能只展示额外头 top-k 命中率。